In [1]:
import pandas as pd
import numpy as np
import re
import pickle as pkl
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
from tqdm import tqdm
import fasttext

In [2]:
train_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/train.parquet")
val_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/val.parquet")
test_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/test.parquet")

In [3]:
train_df.head()["tokenised_unmasked_text"]

0    [dear, hr, ,, please, initiate, the, process, ...
1    [notre, savings, account, doi, ##t, se, confor...
2    [", our, office, ,, located, at, 70, ##7, ##29...
3    [no, ##us, aim, ##eri, ##ons, organise, ##r, u...
4    [to, ensure, we, remain, connected, throughout...
Name: tokenised_unmasked_text, dtype: object

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17330 entries, 0 to 17329
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   masked_text              17330 non-null  object
 1   unmasked_text            17330 non-null  object
 2   token_entity_labels      17330 non-null  object
 3   tokenised_unmasked_text  17330 non-null  object
dtypes: object(4)
memory usage: 541.7+ KB


In [5]:
# from sklearn.model_selection import train_test_split

# train_df, test_df = train_test_split(
#     train_df,
#     test_size=0.08,     
#     random_state=42,    
#     shuffle=True
# )

# train_df, val_df = train_test_split(
#     train_df,
#     test_size=0.1,     
#     random_state=42,    
#     shuffle=True
# )

print(len(train_df), len(val_df),len(test_df)) 

17330 1926 2140


In [6]:
ENWE = fasttext.load_model("/kaggle/input/datasets/bobazooba/fasttext-english/cc.en.300.bin")

In [7]:
all_labels = set()

for labels in train_df["token_entity_labels"]:
    all_labels.update(labels)

label2id = {label: idx for idx, label in enumerate(sorted(all_labels))}
id2label = {idx: label for label, idx in label2id.items()}

print(label2id)
print("Number of classes:", len(label2id))

{'B-ACCOUNTNAME': 0, 'B-ACCOUNTNUMBER': 1, 'B-CREDITCARDNUMBER': 2, 'B-EMAIL': 3, 'B-IPV4': 4, 'B-IPV6': 5, 'B-MAC': 6, 'B-PASSWORD': 7, 'B-PHONE_NUMBER': 8, 'B-SSN': 9, 'B-USERNAME': 10, 'I-ACCOUNTNAME': 11, 'I-ACCOUNTNUMBER': 12, 'I-CREDITCARDNUMBER': 13, 'I-EMAIL': 14, 'I-IPV4': 15, 'I-IPV6': 16, 'I-MAC': 17, 'I-PASSWORD': 18, 'I-PHONE_NUMBER': 19, 'I-SSN': 20, 'I-USERNAME': 21, 'O': 22}
Number of classes: 23


In [8]:
class PiiDataset(Dataset):

    def __init__(self, texts, labels, embedding_model, label2id):
        self.texts = texts
        self.labels = labels
        self.embedding_model = embedding_model
        self.label2id = label2id

        self.max_len = max(len(sent) for sent in texts)

        self.embed_dim = embedding_model.get_dimension()

        self.pad_embedding = torch.zeros(self.embed_dim)

        # ignored by CrossEntropyLoss
        self.pad_label = -100

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        words = self.texts[idx]
        labels = self.labels[idx]

        embeddings = []

        for word in words:
            embeddings.append(
                torch.tensor(
                    self.embedding_model.get_word_vector(word),
                    dtype=torch.float32
                )
            )

        encoded_labels = [
            self.label2id[label]
            for label in labels
        ]

        length = len(words)

        while len(embeddings) < self.max_len:
            embeddings.append(self.pad_embedding)

        while len(encoded_labels) < self.max_len:
            encoded_labels.append(self.pad_label)

        x = torch.stack(embeddings)

        y = torch.tensor(
            encoded_labels,
            dtype=torch.long
        )

        return x, length, y

In [9]:
training_data = PiiDataset(
    np.array(train_df["tokenised_unmasked_text"]),
    np.array(train_df["token_entity_labels"]),
    embedding_model=ENWE,
    label2id=label2id
)

val_data = PiiDataset(
    np.array(val_df["tokenised_unmasked_text"]),
    np.array(val_df["token_entity_labels"]),
    embedding_model=ENWE,
    label2id=label2id
)

test_data = PiiDataset(
    np.array(test_df["tokenised_unmasked_text"]),
    np.array(test_df["token_entity_labels"]),
    embedding_model=ENWE,
    label2id=label2id
)

In [10]:
train_loader = DataLoader(
    training_data,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

In [11]:
class PiiModel(nn.Module):

    def __init__(
        self,
        embedding_dim=300,
        hidden_size=256,
        num_classes=2,
        dropout=0.3
    ):
        super().__init__()

        self.bilstm1 = nn.LSTM(
            embedding_dim,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.bilstm2 = nn.LSTM(
            hidden_size * 2,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.bilstm3 = nn.LSTM(
            hidden_size * 2,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x, lengths):

        # IMPORTANT: NO packing here for debugging stability
        out1, _ = self.bilstm1(x)
        out2, _ = self.bilstm2(out1)
        out3, _ = self.bilstm3(out2)

        logits = self.classifier(out3)

        return logits

In [12]:
def overfit_one_batch(
    model,
    dataset,
    batch_size=32,
    epochs=300,
    lr=1e-2
):

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    x, lengths, y = next(iter(loader))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)

    x = x.to(device)
    y = y.to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()

    for epoch in range(epochs):

        optimizer.zero_grad()

        logits = model(x, lengths)

        # SAFE reshape
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            y.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            mask = (y != -100)

            acc = (preds[mask] == y[mask]).float().mean().item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")

        if acc == 1.0:
            print("Perfect overfit achieved!")
            break

In [13]:
model = PiiModel(
    embedding_dim=300,
    hidden_size=256,
    num_classes=len(label2id),
    dropout=0.0
)

In [14]:
overfit_one_batch(
    model,
    training_data,
    batch_size=32,
    epochs=300,
    lr=1e-2
)

Epoch 0 | Loss: 3.1578 | Acc: 0.0009
Epoch 10 | Loss: 0.5187 | Acc: 0.9200
Epoch 20 | Loss: 0.4538 | Acc: 0.9200
Epoch 30 | Loss: 0.4358 | Acc: 0.9200
Epoch 40 | Loss: 0.4312 | Acc: 0.9200
Epoch 50 | Loss: 0.4462 | Acc: 0.9200
Epoch 60 | Loss: 0.4399 | Acc: 0.9200
Epoch 70 | Loss: 0.4192 | Acc: 0.9200
Epoch 80 | Loss: 0.3238 | Acc: 0.9200
Epoch 90 | Loss: 0.2736 | Acc: 0.9200
Epoch 100 | Loss: 0.2105 | Acc: 0.9465
Epoch 110 | Loss: 0.1465 | Acc: 0.9483
Epoch 120 | Loss: 0.1294 | Acc: 0.9563
Epoch 130 | Loss: 0.1006 | Acc: 0.9679
Epoch 140 | Loss: 0.0859 | Acc: 0.9711
Epoch 150 | Loss: 0.0717 | Acc: 0.9781
Epoch 160 | Loss: 0.0527 | Acc: 0.9809
Epoch 170 | Loss: 0.0446 | Acc: 0.9879
Epoch 180 | Loss: 0.0545 | Acc: 0.9805
Epoch 190 | Loss: 0.0250 | Acc: 0.9921
Epoch 200 | Loss: 0.0136 | Acc: 0.9972
Epoch 210 | Loss: 0.0070 | Acc: 0.9991
Perfect overfit achieved!


In [15]:
model = PiiModel(
    embedding_dim=300,
    hidden_size=256,
    num_classes=len(label2id),
    dropout=0.3
)

In [16]:
def evaluate(model, val_dataloader, pad_label=-100):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=pad_label).to(device)

    total_correct = 0
    total_loss = 0
    total_tokens = 0

    model.eval()

    with torch.no_grad():

        for test_input, test_len, test_label in tqdm(val_dataloader):

            test_input = test_input.to(device)
            test_label = test_label.to(device)
            test_len = test_len.to(device)

            output = model(test_input, test_len)

            loss = criterion(
                output.reshape(-1, output.size(-1)),
                test_label.reshape(-1)
            )

            total_loss += loss.item()

            preds = output.argmax(dim=-1)

            mask = (test_label != pad_label)

            total_correct += ((preds == test_label) & mask).sum().item()
            total_tokens += mask.sum().item()

    final_accuracy = total_correct / total_tokens if total_tokens > 0 else 0
    final_loss = total_loss / len(val_dataloader)

    print(f"\nVal Loss: {final_loss:.4f} | Val Acc: {final_accuracy:.4f}")

    return final_loss, final_accuracy

In [17]:
def train(
    model,
    train_dataset,
    val_dataset,
    batch_size=64,
    epochs=20,
    learning_rate=0.001,
    pad_label=-100,
    patience=3
):

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss(ignore_index=pad_label)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=2,
        gamma=0.5
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):

        total_correct = 0
        total_loss = 0
        total_tokens = 0

        model.train()

        for train_input, train_len, train_label in tqdm(train_loader):

            train_input = train_input.to(device)
            train_label = train_label.to(device)
            train_len = train_len.to(device)

            output = model(train_input, train_len)

            loss = criterion(
                output.reshape(-1, output.size(-1)),
                train_label.reshape(-1)
            )

            total_loss += loss.item()

            preds = output.argmax(dim=-1)

            mask = (train_label != pad_label)

            total_correct += ((preds == train_label) & mask).sum().item()
            total_tokens += mask.sum().item()

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

            optimizer.step()

        scheduler.step()

        train_loss = total_loss / len(train_loader)
        train_acc = total_correct / total_tokens if total_tokens > 0 else 0

        val_loss, val_acc = evaluate(model, val_loader, pad_label=pad_label)

        lr = scheduler.get_last_lr()[0]

        print(
            f"Epoch {epoch+1} | LR: {lr:.6f} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), "pii_model.pth")

        else:
            epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                model.load_state_dict(torch.load("pii_model.pth"))
                break

    return model

In [18]:
trained_model = train(
    model=model,
    train_dataset=training_data,
    val_dataset=val_data,
    batch_size=64,
    epochs=20,
    learning_rate=1e-3,
    pad_label=-100,
    patience=3
)

100%|██████████| 31/31 [00:04<00:00,  7.47it/s]



Val Loss: 0.1673 | Val Acc: 0.9526
Epoch 1 | LR: 0.001000 | Train Loss: 0.3714 | Train Acc: 0.9267 | Val Loss: 0.1673 | Val Acc: 0.9526


100%|██████████| 31/31 [00:04<00:00,  7.60it/s]



Val Loss: 0.0764 | Val Acc: 0.9695
Epoch 2 | LR: 0.000500 | Train Loss: 0.1114 | Train Acc: 0.9619 | Val Loss: 0.0764 | Val Acc: 0.9695


100%|██████████| 31/31 [00:04<00:00,  7.43it/s]


...


Val Loss: 0.0367 | Val Acc: 0.9840
Epoch 10 | LR: 0.000031 | Train Loss: 0.0253 | Train Acc: 0.9886 | Val Loss: 0.0367 | Val Acc: 0.9840


100%|██████████| 31/31 [00:04<00:00,  7.51it/s]



Val Loss: 0.0363 | Val Acc: 0.9845
Epoch 11 | LR: 0.000031 | Train Loss: 0.0240 | Train Acc: 0.9895 | Val Loss: 0.0363 | Val Acc: 0.9845


100%|██████████| 31/31 [00:04<00:00,  7.49it/s]


Val Loss: 0.0359 | Val Acc: 0.9844
Epoch 12 | LR: 0.000016 | Train Loss: 0.0234 | Train Acc: 0.9897 | Val Loss: 0.0359 | Val Acc: 0.9844
Early stopping at epoch 12


In [19]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

def evaluate_test_set(
    model,
    test_dataset,
    label2id,
    batch_size=64,
    pad_label=-100
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    all_preds = []
    all_labels = []

    id2label = {v: k for k, v in label2id.items()}

    with torch.no_grad():

        for x, lengths, y in tqdm(test_loader):

            x = x.to(device)
            y = y.to(device)
            lengths = lengths.to(device)

            logits = model(x, lengths)

            preds = logits.argmax(dim=-1)

            mask = (y != pad_label)

            preds = preds[mask]
            true = y[mask]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(true.cpu().numpy())

    # --- Metrics ---
    f1_micro = f1_score(all_labels, all_preds, average="micro")
    f1_macro = f1_score(all_labels, all_preds, average="macro")

    print("\n===== TEST RESULTS =====")
    print(f"Micro F1: {f1_micro:.4f}")
    print(f"Macro F1: {f1_macro:.4f}")

    print("\n===== CLASSIFICATION REPORT =====")
    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=[id2label[i] for i in sorted(id2label.keys())],
            digits=4
        )
    )

    return f1_micro, f1_macro

In [20]:
evaluate_test_set(
    model=trained_model,
    test_dataset=test_data,
    label2id=label2id,
    batch_size=64,
    pad_label=-100
)

100%|██████████| 34/34 [00:04<00:00,  8.07it/s]



===== TEST RESULTS =====
Micro F1: 0.9831
Macro F1: 0.9171

===== CLASSIFICATION REPORT =====
                    precision    recall  f1-score   support

     B-ACCOUNTNAME     0.9245    0.9333    0.9289       105
   B-ACCOUNTNUMBER     1.0000    0.9550    0.9770       111
B-CREDITCARDNUMBER     0.7723    0.8764    0.8211        89
           B-EMAIL     0.9618    0.9679    0.9649       156
            B-IPV4     0.7353    0.9921    0.8446       126
            B-IPV6     0.7426    0.9439    0.8313       107
             B-MAC     1.0000    1.0000    1.0000        84
        B-PASSWORD     0.9694    0.9314    0.9500       102
    B-PHONE_NUMBER     1.0000    0.9907    0.9953       107
             B-SSN     0.9787    0.9892    0.9840        93
        B-USERNAME     0.8304    0.6327    0.7181       147
     I-ACCOUNTNAME     0.9101    0.9609    0.9348       179
   I-ACCOUNTNUMBER     0.9947    0.9691    0.9817       388
I-CREDITCARDNUMBER     0.7626    0.8813    0.8176       758
    

(0.9830852060945572, 0.9170648822978333)